# Week 5 — Window Functions: ROW_NUMBER, RANK & QUALIFY
**Phase 2: Intermediate · ~15 minutes · Tables: `OLIST_ORDERS`, `OLIST_ORDER_ITEMS`, `OLIST_ORDER_PAYMENTS`**

---

## What you're practicing this week

Window functions let you calculate across a set of related rows without collapsing them into groups. Unlike `GROUP BY`, the original rows are preserved — you just get an extra calculated column alongside them. They're one of the most powerful tools in SQL analytics.

## New concepts this week

The syntax for every window function follows the same pattern:

```sql
FUNCTION_NAME() OVER (
    PARTITION BY column  -- optional: restart the calculation per group
    ORDER BY column      -- required for ranking/running functions
)
```

| Function | What it does |
|---|---|
| `ROW_NUMBER()` | Unique sequential number — no ties, always 1,2,3,4... |
| `RANK()` | Tied values get the same rank, then skips — 1,1,3,4... |
| `DENSE_RANK()` | Tied values get the same rank, no skipping — 1,1,2,3... |
| `PERCENT_RANK()` | Position as a percentage from 0 to 1 |
| `NTILE(n)` | Divides rows into n equal buckets |

### QUALIFY — Snowflake's shortcut
Normally to filter on a window function result you'd need a subquery. Snowflake's `QUALIFY` clause lets you do it inline:

```sql
-- Without QUALIFY (verbose subquery)
SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_purchase_timestamp DESC) AS rn
    FROM OLIST_ORDERS
) WHERE rn = 1;

-- With QUALIFY (Snowflake only — much cleaner)
SELECT *,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_purchase_timestamp DESC) AS rn
FROM OLIST_ORDERS
QUALIFY rn = 1;
```

---

## Try the examples

Run these to see window functions in action.

In [ ]:
-- Example 1: rank every order by payment value (highest = rank 1)
SELECT
    order_id,
    payment_value,
    RANK() OVER (ORDER BY payment_value DESC) AS payment_rank
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDER_PAYMENTS
ORDER BY payment_rank
LIMIT 10;

In [ ]:
-- Example 2: rank orders within each payment type
-- PARTITION BY restarts the ranking for each group
SELECT
    order_id,
    payment_type,
    payment_value,
    RANK() OVER (PARTITION BY payment_type ORDER BY payment_value DESC) AS rank_within_type
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDER_PAYMENTS
QUALIFY rank_within_type <= 3
ORDER BY payment_type, rank_within_type;

In [ ]:
-- Example 3: get each customer's most recent order using ROW_NUMBER + QUALIFY
SELECT
    customer_id,
    order_id,
    order_purchase_timestamp,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_purchase_timestamp DESC) AS rn
FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS
QUALIFY rn = 1
LIMIT 10;

---

## Explore


In [ ]:
SELECT * FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS LIMIT 5;

---

## Task 1 — Rank orders by payment value per payment type

Using `OLIST_ORDER_PAYMENTS`, rank each order by `payment_value` within each `payment_type`. Return `order_id`, `payment_type`, `payment_value`, and `rank_within_type`. Use `RANK()` ordered by payment_value descending.

> **What to think about:** What does PARTITION BY payment_type do here? Remove it and see how the results change.

In [ ]:
-- Task 1: your query here


In [ ]:
-- Submit Task 1
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    5,
    1,
    $$
        -- paste your Task 1 query here
    $$
);

---

## Task 2 — Each customer's most recent order

For every customer, return only their most recent order. Use `ROW_NUMBER()` with `QUALIFY` to keep only the first row per customer when ordered by `order_purchase_timestamp` descending. Return `customer_id`, `order_id`, and `order_purchase_timestamp`.

> **What to think about:** Why is `ROW_NUMBER()` better than `RANK()` here? What would happen if two orders had identical timestamps and you used RANK?

In [ ]:
-- Task 2: your query here


In [ ]:
-- Submit Task 2
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    5,
    2,
    $$
        -- paste your Task 2 query here
    $$
);

---

## Task 3 — Top 3 items by price per order

Using `OLIST_ORDER_ITEMS`, for each order return the top 3 most expensive items by `price`. Use `ROW_NUMBER()` with `QUALIFY`. Return `order_id`, `product_id`, `price`, and `price_rank`.

> **What to think about:** This pattern — top N rows per group — is one of the most common uses of window functions in analytics.

In [ ]:
-- Task 3: your query here


In [ ]:
-- Submit Task 3
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    5,
    3,
    $$
        -- paste your Task 3 query here
    $$
);

---

## Task 4 — Percentile rank of orders by payment value

Using `OLIST_ORDER_PAYMENTS`, calculate the `PERCENT_RANK()` of each order by `payment_value`. Return `order_id`, `payment_value`, and `percentile_rank` rounded to 2 decimal places. Sort by payment_value descending.

> **What to think about:** A PERCENT_RANK of 0.95 means the order is in the top 5% by value. How would you filter to only show the top decile (top 10%)?

In [ ]:
-- Task 4: your query here


In [ ]:
-- Submit Task 4
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    5,
    4,
    $$
        -- paste your Task 4 query here
    $$
);

---

## Bonus challenge

For each customer, find their single highest-value order. Use a CTE to calculate total payment per order first, then apply a window function with QUALIFY to keep only the top order per customer.

In [ ]:
-- Bonus: your query here


---

*Next week: LAG, LEAD, and running totals — working with sequences of rows over time.*